# n(z) data challenge: local results on a held-out split

A local rehearsal of the n(z) challenge (LSSTDESC `nz_data_challenge`) before the final submission. The challenge gives, per configuration, a WFD catalogue without redshifts (the target) and 5 DDF fields with redshifts (the training data), and scores a submission on the WFD truth, which is hidden. Here the labelled DDF fields play both roles: they are pooled and split at random, and ZNN is trained on 80% and scored on the other 20% (`VAL_FRAC`, `SEED`) exactly as the challenge scores the WFD, with its own metric code (`nz_data_challenge.evaluation`).

The held-out split is drawn from the same DDF fields as the training split, so the scores are **optimistic**: they ignore the differences between the DDF and the WFD (depth, field-to-field variance). They indicate how well the library does, not the final score.

For each configuration (task sets 1 and 2 × Cardinal and Flagship × 1yr and 4yr):

- **Training** (only if the model file is missing): F and G (`znn.train_model`) on the 80%, saved to `OUT_DIR`.
- **Estimates**, in the submission format, for the held-out galaxies:
  - `tomo_bin_index`: the tomographic bin of each galaxy's p(z) mode, with the challenge's bin edges;
  - the n(z) of each tomographic bin: the stacked p(z) of its galaxies, a qp ensemble with `n_objects` in its ancil;
  - n(z) realizations from the copula (`znn.fit_copula`, `znn.sample_redshifts`), fitted on the training split.
- **Scores**: the metrics of `evaluation.evaluate_submission` (bin assignment, n(z) summary statistics, Fisher forecasts against perfect tomography) and their challenge points; and the χ² of the true n(z) against the copula realizations.

Run from the repo root, with the `nz_data_challenge` repo next to it and its public data in `../nz_data_challenge/public`.

## Configuration

In [ ]:
import sys

import znn  # run from the repo root so this picks up the local source

sys.path.insert(0, "../nz_data_challenge/src")  # the challenge's own metric code
from nz_data_challenge import evaluation, forecast, utils

TASKSETS = utils.TASKSETS  # the scored task sets
SIMS = utils.SIMS
SCENARIOS = utils.SCENARIOS
DATA_DIR = "../nz_data_challenge/public"
N_DDF = 5  # DDF fields per configuration

PRETRAINING = "popcosmos"  # F starts from its pop-cosmos pre-training (None: from scratch)
NU = znn.DEFAULT_NU  # degrees of freedom of G's Student-t
EPOCHS = 100  # max epochs of F and of G (early stopping usually ends sooner)
OUT_DIR = (f"outputs/znn_nz" if PRETRAINING is None else f"outputs/znn_nz_pretrained_{PRETRAINING}") + f"_moment_t{NU:g}"
CACHE_DIR = f"{OUT_DIR}_predictions"  # held-out estimates and n(z) realizations
PLOT_DIR = "plots"

# the held-out split of the pooled DDF fields
VAL_FRAC = 0.2
SEED = 42

N_SAMPLES = 100  # n(z) realizations per tomographic bin, as in the submission's nz_samples
N_VAL_DRAWS = 500  # copula realizations for the chi^2
COPULA_SEED = 0
NZ_CHI2_BINS = 0.05  # width of the n(z) bins of the chi^2
NZ_MIN_COUNT = 20  # n(z) bins in the chi^2: expected to hold at least this many held-out galaxies

In [ ]:
import itertools
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import qp
from rail.utils.path_utils import find_rail_file
from sklearn.model_selection import train_test_split

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

CONFIGS = list(itertools.product(TASKSETS, SIMS, SCENARIOS))
TASKSET_COLORS = {"taskset_1": plt.get_cmap("Blues")(0.7), "taskset_2": plt.get_cmap("Greens")(0.7)}
SIM_LABELS = {"cardinal": "Cardinal", "flagship": "Flagship"}
DEPTH_LABELS = {"1yr": "Y1", "4yr": "Y4"}

# the bands, as in the p(z) task-set notebooks: catalogue column and RAIL filter curve, by wavelength
FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"
FEATURE_CONFIG = znn.DEFAULT_FEATURE_CONFIG


def label(taskset, sim, scenario):
    return f"{taskset.replace('taskset_', 'TS')} {SIM_LABELS[sim]} {DEPTH_LABELS[scenario]}"


def model_file(taskset, sim, scenario):
    return os.path.join(OUT_DIR, f"nz_challenge_{taskset}_{sim}_{scenario}_model.pkl")


def save(fig, name):
    fig.savefig(os.path.join(PLOT_DIR, f"{os.path.basename(OUT_DIR)}_{name}.pdf"), bbox_inches="tight")

## Data and split

The 5 DDF fields of a configuration are pooled (the columns they share) and split at random into 80% for training and 20% held out. The reference redshift is `redshift`, filled with `redshift_manyband` where it is missing, as in the challenge's cross-validation scripts; galaxies with neither are dropped.

In [ ]:
def ddf_split(taskset, sim, scenario):
    # (train, z_train), (held-out, z_held_out) of the pooled DDF fields
    ddfs = [znn.read_catalog(f"{DATA_DIR}/nz_challenge_{taskset}_{sim}_{scenario}_ddf_{i:02d}.hdf5")
            for i in range(N_DDF)]
    df = pd.concat(ddfs, join="inner", ignore_index=True)
    z = df["redshift"].to_numpy(dtype=float).copy()
    if "redshift_manyband" in df:
        missing = ~np.isfinite(z)
        z[missing] = df["redshift_manyband"].to_numpy(dtype=float)[missing]
    keep = np.isfinite(z)
    df, z = df[keep].reset_index(drop=True), z[keep]
    idx_train, idx_val = train_test_split(np.arange(len(df)), test_size=VAL_FRAC, random_state=SEED)
    return (df.iloc[idx_train], z[idx_train]), (df.iloc[idx_val], z[idx_val])

## Train the models (only if needed)

For each configuration whose model file is not in `OUT_DIR`: F on the training split (early stopping on its training loss), from the pop-cosmos pre-training, then G on F's residuals with F fixed. Delete a model file to retrain it, and its cached estimates in `CACHE_DIR` with it.

In [ ]:
for c in CONFIGS:
    if os.path.exists(model_file(*c)):
        print("found", model_file(*c))
        continue
    print(f"\n========== training: {label(*c)} ==========")
    (train, z_train), _ = ddf_split(*c)
    model, _ = znn.train_model(znn.build_model, train, BANDS, REF_BAND, FEATURE_CONFIG, Y=z_train, nu=NU,
                               EPOCHS=EPOCHS, pretraining=PRETRAINING)
    znn.save_model_file(model_file(*c), model, BANDS, REF_BAND, FEATURE_CONFIG)
    print("saved", model_file(*c))

## Held-out estimates in the submission format

For the held-out galaxies of each configuration:

- `tomo_bin_index`: the tomographic bin of the p(z) mode (`utils.TOMO_BIN_EDGES`; a mode beyond the last edge goes into the last bin, as the challenge assigns the true bins);
- the n(z) of each tomographic bin: the stacked p(z) of its galaxies (the mean, over them, of the p(z) mass in each bin of a 0.01-wide grid, from the CDF), a `qp.hist` ensemble with `n_objects` in its ancil;
- n(z) realizations: `N_VAL_DRAWS` joint draws of the held-out redshifts with the copula fitted on the training split, each histogrammed per tomographic bin (the first `N_SAMPLES` are what the submission's `nz_samples` would hold).

They are cached in `CACHE_DIR`, and recomputed when the model file is newer or the copula's ρ changed.

In [ ]:
STACK_EDGES = np.linspace(0, 3, 301)  # grid of the stacked n(z)
CHI2_EDGES = np.arange(0, 3 + NZ_CHI2_BINS / 2, NZ_CHI2_BINS)


def tomo_index(zmode, taskset):
    return np.digitize(zmode, utils.TOMO_BIN_EDGES[taskset][1:-1])


def heldout_estimates(taskset, sim, scenario):
    c = (taskset, sim, scenario)
    cache = os.path.join(CACHE_DIR, f"nz_challenge_{taskset}_{sim}_{scenario}_heldout.npz")
    model, features = znn.load_model_file(model_file(*c))
    (train, z_train), (val, z_val) = ddf_split(*c)
    copula = znn.fit_copula(model, train, **features, Y=z_train)
    if os.path.exists(cache) and os.path.getmtime(cache) >= os.path.getmtime(model_file(*c)):
        cached = dict(np.load(cache))
        if np.isclose(cached["rho"], copula["rho"], rtol=0, atol=1e-10):
            return cached
    n_tomo = len(utils.TOMO_BIN_EDGES[taskset]) - 1
    pz = znn.predict(model, val, **features)
    zmode = np.squeeze(pz.ancil["zmode"])
    tomo = tomo_index(zmode, taskset)
    p_bin = np.diff(pz.cdf(STACK_EDGES), axis=1)  # p(z) mass of every galaxy in every grid bin
    stacked = np.array([p_bin[tomo == k].mean(axis=0) for k in range(n_tomo)])
    zs = znn.sample_redshifts(model, copula, val, **features, n_draws=N_VAL_DRAWS,
                              rng=np.random.default_rng(COPULA_SEED))
    realizations = np.array([[np.histogram(z[tomo == k], CHI2_EDGES)[0] for k in range(n_tomo)] for z in zs])
    out = dict(rho=copula["rho"], z_true=z_val, zmode=zmode, tomo=tomo, stacked=stacked,
               n_objects=np.bincount(tomo, minlength=n_tomo), realizations=realizations)
    np.savez(cache, **out)
    return out


estimates = {}
for c in CONFIGS:
    print("held-out estimates:", label(*c))
    estimates[c] = heldout_estimates(*c)

## Challenge metrics

The metrics of `evaluation.evaluate_submission`, computed on the held-out split as the challenge computes them on the WFD: bin assignment (accuracy, balanced accuracy, Cohen's κ, log loss, mutual information), n(z) (RMS over tomographic bins of the error of the mean and width of each n(z), total information loss), and Fisher forecasts (cosmic shear σ(S₈) and 3x2pt FoM(w₀, w_a) against perfect tomography, and the biases from the error of the n(z)).

The effective number densities of the tomographic bins are those of the full WFD: the challenge converts the counts of its 1M WFD objects, so the held-out counts are scaled up to 1M first. Every configuration runs 4 Fisher forecasts.

In [ ]:
def perfect_forecast(taskset, sim, scenario, mode, z_true):
    # evaluation.perfect_forecast, on an array of true redshifts rather than a truth file
    edges = utils.TOMO_BIN_EDGES[taskset]
    n_tomo = len(edges) - 1
    true_tomo = utils.get_true_bin_assignments(z_true, edges)
    nz_perfect = utils.get_true_nz_distributions(z_true, true_tomo, forecast.FORECAST_Z_GRID_FULL, n_tomo)
    counts = np.bincount(true_tomo, minlength=n_tomo) * forecast.N_SAMPLED_OBJ / len(z_true)
    flex = evaluation.FisherFlex(
        nz_source=nz_perfect, z_grid=forecast.FORECAST_Z_GRID,
        neff_source=forecast.tomo_bins_effective_density(counts, taskset, sim, scenario),
        fsky=forecast.FORECAST_FSKY, sigma_e=forecast.FORECAST_SIGMA_E, mode=mode, nz_model="no_uncertainty",
    )
    flex.compute(parallel=True)
    return flex.forecast(**evaluation.FISHER_FORECAST_PARAMS)


def challenge_metrics(taskset, sim, scenario):
    e = estimates[(taskset, sim, scenario)]
    edges = utils.TOMO_BIN_EDGES[taskset]
    grid_edges = utils.Z_BIN_EDGES[taskset]
    n_tomo = len(edges) - 1
    z_true, tomo = e["z_true"], e["tomo"]
    nz_estimates = qp.Ensemble(qp.hist, data=dict(bins=STACK_EDGES, pdfs=e["stacked"]),
                               ancil=dict(n_objects=e["n_objects"]))
    bhat = dict(tomo_bin_index=tomo)
    truth = dict(redshift=z_true)

    out = evaluation.evaluate_bin_assignments(utils.get_true_bin_assignments(z_true, edges), tomo)
    grid_centers = 0.5 * (grid_edges[1:] + grid_edges[:-1])
    out.update(evaluation.evaluate_distributions(
        utils.get_true_nz_distributions(z_true, tomo, grid_edges, n_tomo),
        utils.get_nz_distributions(nz_estimates, grid_centers, n_tomo), grid_edges, e["n_objects"]))
    neff = forecast.tomo_bins_effective_density(e["n_objects"] * forecast.N_SAMPLED_OBJ / len(z_true),
                                                taskset, sim, scenario)
    res_cs, bias_cs = forecast.fisher_bias_forecast(nz_estimates, bhat, neff, mode="cosmic_shear", truth=truth)
    res_3x2, bias_3x2 = forecast.fisher_bias_forecast(nz_estimates, bhat, neff, mode="3x2pt", truth=truth)
    out.update(evaluation.evaluate_fisher_forecasts(
        res_cs, bias_cs, res_3x2, bias_3x2,
        perfect_forecast(taskset, sim, scenario, "cosmic_shear", z_true),
        perfect_forecast(taskset, sim, scenario, "3x2pt", z_true)))
    return out


cache = os.path.join(CACHE_DIR, "challenge_metrics.pkl")
rhos = {c: float(estimates[c]["rho"]) for c in CONFIGS}
if os.path.exists(cache) and pd.read_pickle(cache).attrs.get("rho") == rhos and all(
        os.path.getmtime(cache) >= os.path.getmtime(model_file(*c)) for c in CONFIGS):
    metrics_table = pd.read_pickle(cache)
else:
    metrics_table = pd.DataFrame({c: challenge_metrics(*c) for c in CONFIGS}).T
    metrics_table.index.names = ["taskset", "sim", "scenario"]
    metrics_table.attrs["rho"] = rhos
    metrics_table.to_pickle(cache)

SCORED = list(evaluation.METRICS)
display(metrics_table[SCORED].astype(float).style.format(precision=3))

### Challenge points

Each metric graded against the challenge's tiers (`evaluation.METRICS`): one point per nested range the value falls in, so at most 3 per metric.

In [ ]:
points = evaluation.build_scores_dataframe(metrics_table[SCORED].astype(float).reset_index())
points = points.set_index(["taskset", "sim", "scenario"])[SCORED]
points["total"] = points.sum(axis=1)
print(f"points per configuration, out of {3 * len(SCORED)}")
display(points)
print("total per task set:")
print(points["total"].groupby(level="taskset").sum().to_string())

## n(z) and its uncertainty

The true n(z) of each tomographic bin of the held-out galaxies (black) against the mean of the copula realizations, with their spread as error bars (coloured), in bins of `NZ_CHI2_BINS`. The table gives χ²/ν of the true n(z) of all tomographic bins together against the mean and covariance of the realizations, over the n(z) bins expected to hold at least `NZ_MIN_COUNT` galaxies, with the Hartlap correction: about 1 if the realizations describe the error of the n(z). The submission's n(z) samples are not used by the challenge's current scoring (its Fisher forecasts take the n(z) as exactly known), so this is the check of the copula.

In [ ]:
def chi2(truth, realizations):
    # chi^2 of the truth against the mean and covariance of the realizations (rows), with the Hartlap correction
    r = truth - realizations.mean(0)
    n, dof = realizations.shape
    return (n - dof - 2) / (n - 1) * r @ np.linalg.solve(np.atleast_2d(np.cov(realizations, rowvar=False)), r)


rows = {}
for c in CONFIGS:
    e = estimates[c]
    n_tomo = e["realizations"].shape[1]
    truth = np.array([np.histogram(e["z_true"][e["tomo"] == k], CHI2_EDGES)[0] for k in range(n_tomo)]).ravel()
    real = e["realizations"].reshape(len(e["realizations"]), -1).astype(float)
    used = real.mean(0) >= NZ_MIN_COUNT
    rows[c] = {"ρ": float(e["rho"]), "n(z) bins": int(used.sum()), "χ²/ν": chi2(truth[used], real[:, used]) / used.sum()}
nz_chi2 = pd.DataFrame(rows).T
nz_chi2.index.names = ["taskset", "sim", "scenario"]
display(nz_chi2.style.format({"ρ": "{:.3f}", "n(z) bins": "{:d}", "χ²/ν": "{:.2f}"}))

centres = 0.5 * (CHI2_EDGES[1:] + CHI2_EDGES[:-1])
rows_ = list(itertools.product(SIMS, SCENARIOS))
fig, axes = plt.subplots(len(rows_), len(TASKSETS), figsize=(6 * len(TASKSETS), 2.6 * len(rows_)), sharex=True,
                         gridspec_kw=dict(hspace=0, wspace=0.15))
for col, taskset in enumerate(TASKSETS):
    cmap = plt.get_cmap("viridis")
    for row, (sim, scenario) in enumerate(rows_):
        e, ax = estimates[(taskset, sim, scenario)], axes[row, col]
        n_tomo = e["realizations"].shape[1]
        for k in range(n_tomo):
            truth = np.histogram(e["z_true"][e["tomo"] == k], CHI2_EDGES)[0]
            ax.stairs(truth, CHI2_EDGES, color="k", lw=0.8)
            real = e["realizations"][:, k]
            ax.errorbar(centres, real.mean(0), real.std(0, ddof=1), fmt="o", ms=2, lw=1, capsize=0,
                        color=cmap(k / max(n_tomo - 1, 1)))
        ax.text(0.98, 0.92, f"{label(taskset, sim, scenario)}   χ²/ν {nz_chi2.loc[(taskset, sim, scenario), 'χ²/ν']:.2f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=9)
        if row == len(rows_) - 1:
            ax.set_xlabel("z")
        if col == 0:
            ax.set_ylabel("galaxies per bin")
axes[0, 0].set_xlim(0, 3)
save(fig, "nz_heldout")
plt.show()